# Ingest the NYC TLC taxi zone shape files
NYC TLC trip record data (https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) consists of a lookup table with shape files, which can be used for visualizing maps. The zipped file has been downloaded to `/Volumes/dbx_joshdevph_dev/raw/vlm` volume. Files included in the zipped file are as follows. 
- taxi_zones.cpg
- taxi_zones.prj
- taxi_zones.shp
- taxi_zones.shx

In [0]:
%pip install geopandas fsspec --quiet

In [0]:
import geopandas
import fsspec

# Read shape file from the volume
gdf = geopandas.read_file("/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/taxi_zones/taxi_zones.shp")

# Convert it to WGS84
gdf = gdf.to_crs(epsg=4326) 

# Convert the geometry column to WKT
gdf["geometry_wkt"] = gdf.geometry.to_wkt() 

# Drop geometry column
gdf_spark = gdf.drop(columns=["geometry"])

In [0]:
%sql 
DROP TABLE IF EXISTS dbx_joshdevph_dev.raw.stg_taxi_zone_shape;

In [0]:
zones_df = spark.createDataFrame(gdf_spark)

# Write to Delta table
target_table = "dbx_joshdevph_dev.raw.stg_taxi_zone_shape"
zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)